# AVION + SMS Loss — EPIC-KITCHENS-100 MIR

Replicates the 2024 winning approach (~74% nDCG) using:
- **AVION** backbone (CLIP ViT-L pre-trained on Ego4D via LaViLa)
- **SMS Loss** (Symmetric Multi-Similarity Loss with relevancy matrix)

## Prerequisites — do these BEFORE running any cell

### 1. Runtime: GPU → A100
Runtime → Change runtime type → A100

### 2. Add AVION pre-processed videos to your Drive
Open this link and click **"Add shortcut to Drive"** → My Drive:
```
https://drive.google.com/file/d/13J2uC2g2H_DEHrBvgr5Aiu0BgqlCvWqG/view
```
If it is a zip file, also run the extraction cell (Cell 5b). The folder must be named `EK100_320p_15sec_30fps_libx264`.

### 3. Upload annotation files to your Drive
From your local machine, upload these files to `MyDrive/EK100_annotations/`:
```
EK100_MIR/data/MI-MM/dataframes/EPIC_100_retrieval_train.csv
EK100_MIR/data/MI-MM/dataframes/EPIC_100_retrieval_test.csv
EK100_MIR/data/MI-MM/relevancy/caption_relevancy_EPIC_100_retrieval_train.pkl
```
The test relevancy (`caption_relevancy_EPIC_100_retrieval_test.pkl`) is bundled
inside the AVION GDrive archive. If missing, local nDCG metrics will be skipped
but the submission file is still generated.

In [ ]:
# Cell 1 — GPU check
!nvidia-smi
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2 — Install dependencies
# SMS-Loss requirements + decord for video decoding
!pip install -q ninja==1.11.1 einops==0.8.0 kornia==0.6.10 \
    pandas==1.5.3 scikit-learn==1.5.0 timm==1.0.7 \
    transformers==4.42.3 decord
!pip install -q git+https://github.com/openai/CLIP.git
print("Dependencies installed.")

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Cell 4 — Configure paths
import os

GDRIVE = "/content/drive/MyDrive"

# Path to the AVION pre-processed EK-100 videos (320p, 15-sec chunks)
# Must contain participant folders: P01/, P02/, ..., P37/
DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"

# Annotation files — upload from local EK100_MIR/data/ before running
ANNOT_DIR = f"{GDRIVE}/EK100_annotations"

# Experiment output (saved to Drive so it survives session disconnect)
EXP_DIR = f"{GDRIVE}/experiments/sms_vitl"

# AVION ViT-L pretrain checkpoint (downloaded in Cell 6)
PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

os.makedirs(ANNOT_DIR, exist_ok=True)
os.makedirs(EXP_DIR, exist_ok=True)
os.makedirs(os.path.dirname(PRETRAIN_CKPT), exist_ok=True)

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"  exists  : {os.path.isdir(DATA_ROOT)}")
print(f"ANNOT_DIR : {ANNOT_DIR}")
print(f"EXP_DIR   : {EXP_DIR}")

In [ ]:
# Cell 5a — Check video data
# The AVION GDrive link might be a folder shortcut OR a zip file.
# After adding the shortcut, check whether it's already a directory:
if os.path.isdir(DATA_ROOT):
    participants = [d for d in os.listdir(DATA_ROOT) if d.startswith('P')]
    print(f"Found {len(participants)} participant folders: {sorted(participants)[:5]}...")
else:
    print("DATA_ROOT not found as a directory.")
    # Check if the shortcut landed as a zip:
    zip_path = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264.zip"
    if os.path.isfile(zip_path):
        print(f"Found zip at {zip_path}. Run Cell 5b to extract.")
    else:
        print("Neither folder nor zip found. Please add the GDrive shortcut first.")
        print("Link: https://drive.google.com/file/d/13J2uC2g2H_DEHrBvgr5Aiu0BgqlCvWqG/view")

In [ ]:
# Cell 5b — Extract zip (SKIP if Cell 5a already found the folder)
# Only run this if the GDrive file is a zip
zip_path = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264.zip"
if os.path.isfile(zip_path):
    print(f"Extracting {zip_path} ...")
    !unzip -q "$zip_path" -d "$GDRIVE/"
    print("Done.")
else:
    print("No zip found — skipping.")

In [ ]:
# Cell 6 — Download AVION LaViLa ViT-L pretrain checkpoint (~1.3 GB)
# Source: https://github.com/zhaoyue-zephyrus/AVION/blob/main/scripts/download_checkpoints.sh
if not os.path.isfile(PRETRAIN_CKPT):
    print("Downloading AVION ViT-L pretrain checkpoint...")
    !wget --show-progress -O "$PRETRAIN_CKPT" \
        "https://utexas.box.com/shared/static/1iatmrs7ufdeooce09a61t1n6wsouf4l.pt"
else:
    print(f"Checkpoint already present ({os.path.getsize(PRETRAIN_CKPT)/1e9:.2f} GB).")

In [ ]:
# Cell 7 — Verify annotation files
# These should have been uploaded from local EK100_MIR/data/ to ANNOT_DIR on Drive.
# If any are missing, this cell downloads them from EPIC-KITCHENS GitHub.
import subprocess

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
# Test relevancy is bundled inside the AVION GDrive archive under:
#   epic-kitchens-100-annotations/retrieval_annotations/relevancy/
# Try that path first, then fall back to ANNOT_DIR.
TEST_REL_AVION = (
    f"{DATA_ROOT}/epic-kitchens-100-annotations/"
    "retrieval_annotations/relevancy/"
    "caption_relevancy_EPIC_100_retrieval_test.pkl"
)
TEST_REL = TEST_REL_AVION if os.path.isfile(TEST_REL_AVION) else f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

BASE = "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/retrieval_annotations"

for path, url in [
    (TRAIN_CSV, f"{BASE}/EPIC_100_retrieval_train.csv"),
    (TEST_CSV,  f"{BASE}/EPIC_100_retrieval_test.csv"),
]:
    if not os.path.isfile(path):
        print(f"Downloading {os.path.basename(path)} ...")
        subprocess.run(["wget", "-q", "-O", path, url], check=True)

# Train relevancy — GitHub raw works for pkl
if not os.path.isfile(TRAIN_REL):
    url = f"{BASE}/relevancy/caption_relevancy_EPIC_100_retrieval_train.pkl"
    print("Downloading train relevancy ...")
    subprocess.run(["wget", "-q", "-O", TRAIN_REL, url], check=True)

print(f"TRAIN_CSV  : {os.path.isfile(TRAIN_CSV)}")
print(f"TEST_CSV   : {os.path.isfile(TEST_CSV)}")
print(f"TRAIN_REL  : {os.path.isfile(TRAIN_REL)}")
print(f"TEST_REL   : {os.path.isfile(TEST_REL)}  (path: {TEST_REL})")

In [ ]:
# Cell 8 — Clone SMS-Loss repo
import os
os.chdir('/content')
if not os.path.isdir('/content/SMS-Loss'):
    !git clone https://github.com/xqwang14/SMS-Loss.git
os.chdir('/content/SMS-Loss')
print("Working directory:", os.getcwd())
!ls scripts/

In [ ]:
# Cell 9 — Fine-tune AVION ViT-L with SMS Loss  (single A100, ~10-14 h for 50 epochs)
#
# Key settings vs. the 4-GPU paper config:
#   batch-size 16 × update-freq 4  →  effective batch = 64  (same as paper's 60×4=240 scaled down)
#   epochs 50  (paper uses 100; ViT-L is already pre-trained so 50 is usually sufficient)
#   fused-decode-crop removed  (requires custom decord build; standard decord works fine)
#
# Outputs checkpoint_best.pt to EXP_DIR on Google Drive.

import os
os.chdir('/content/SMS-Loss')

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
TEST_REL_AVION = (
    f"{DATA_ROOT}/epic-kitchens-100-annotations/"
    "retrieval_annotations/relevancy/"
    "caption_relevancy_EPIC_100_retrieval_test.pkl"
)
TEST_REL = TEST_REL_AVION if os.path.isfile(TEST_REL_AVION) else f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"
EXP_DIR       = f"{GDRIVE}/experiments/sms_vitl"

cmd = f"""\
torchrun --nproc_per_node=1 scripts/ammplus_finetune.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --relevancy-train "{TRAIN_REL}" \\
  --relevancy-test  "{TEST_REL}" \\
  --pretrain-model  "{PRETRAIN_CKPT}" \\
  --model CLIP_VITL14 \\
  --batch-size 16 \\
  --update-freq 4 \\
  --epochs 50 \\
  --lr 2e-5 \\
  --loss-margin 0.6 \\
  --loss-thres  0.1 \\
  --use-flash-attn \\
  --grad-checkpointing \\
  --output-dir "{EXP_DIR}"
"""
print(cmd)
!{cmd}

In [ ]:
# Cell 10 — Generate submission pickle with test-time flip augmentation
#
# Outputs test4.pkl to EXP_DIR. The --flip flag averages normal + horizontally
# flipped video embeddings for a small nDCG boost (~+0.3%).

import os, glob
os.chdir('/content/SMS-Loss')

# Use best checkpoint; fall back to latest if best not saved
best_ckpt = f"{EXP_DIR}/checkpoint_best.pt"
if not os.path.isfile(best_ckpt):
    ckpts = sorted(glob.glob(f"{EXP_DIR}/checkpoint_*.pt"))
    best_ckpt = ckpts[-1] if ckpts else None
    print(f"checkpoint_best.pt not found, using: {best_ckpt}")

assert best_ckpt and os.path.isfile(best_ckpt), f"No checkpoint found in {EXP_DIR}"

PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"
TEST_REL_AVION = (
    f"{DATA_ROOT}/epic-kitchens-100-annotations/"
    "retrieval_annotations/relevancy/"
    "caption_relevancy_EPIC_100_retrieval_test.pkl"
)
TEST_REL = TEST_REL_AVION if os.path.isfile(TEST_REL_AVION) else f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

rel_arg = f'--relevancy-path "{TEST_REL}"' if os.path.isfile(TEST_REL) else ""

cmd = f"""\
torchrun --nproc_per_node=1 scripts/test_mir.py \\
  --root "{DATA_ROOT}" \\
  --val-metadata "{TEST_CSV}" \\
  {rel_arg} \\
  --pretrain-model "{best_ckpt}" \\
  --model CLIP_VITL14 \\
  --flip \\
  --use-flash-attn \\
  --batch-size 32 \\
  --output-dir "{EXP_DIR}"
"""
print(cmd)
!{cmd}

In [ ]:
# Cell 11 — Package submission zip for Codabench upload
import os, zipfile, shutil

PKL_SRC  = f"{EXP_DIR}/test4.pkl"
ZIP_OUT  = f"{EXP_DIR}/submission_sms_avion.zip"

assert os.path.isfile(PKL_SRC), f"test4.pkl not found at {PKL_SRC}"

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(PKL_SRC, arcname='test.pkl')  # Codabench expects 'test.pkl' inside

size_mb = os.path.getsize(ZIP_OUT) / 1e6
print(f"Submission zip: {ZIP_OUT}")
print(f"Size: {size_mb:.1f} MB")
print("\nDownload and submit to: https://www.codabench.org/competitions/12008/")

In [ ]:
# Cell 12 — (Optional) Download submission zip from Colab to local machine
from google.colab import files
files.download(ZIP_OUT)